# Naive RAG Implementation

This notebook implements a standard RAG pipeline:
1. Load chunks from Phase 1
2. Create embeddings using OpenAI
3. Build FAISS vector store
4. Implement retrieval pipeline
5. Implement generation pipeline
6. Evaluate on 20 QA pairs

In [1]:
import os
import json
import numpy as np
import faiss
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

## 1. Load Data from Phase 1

In [2]:
# Load chunks
with open('data/chapter8_chunks.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print(f'Loaded {len(chunks)} chunks')
print(f'Sample chunk: {chunks[0]}')

Loaded 170 chunks
Sample chunk: {'chunk_id': 0, 'text': 'Speech and Language Processing.\nDaniel Jurafsky & James H. Martin.\nCopyright © 2026.\nAll\nrights reserved.\nDraft of January 6, 2026.\nCHAPTER\n8\nTransformers\n“The true art of memory is the art of attention ”\nSamuel Johnson, Idler #74, September 1759\nIn this chapter we introduce the transformer, the standard architecture for build-\ning large language models. As we discussed in the prior chapter, transformer-based\nlarge language models have completely changed the ﬁeld of speech and language', 'page': 0, 'source': '8.pdf'}


In [3]:
# Load QA pairs
with open('data/chapter8_qa_pairs.json', 'r', encoding='utf-8') as f:
    qa_pairs = json.load(f)

print(f'Loaded {len(qa_pairs)} QA pairs')
print(f'\nFirst QA pair:')
print(f'Q: {qa_pairs[0]["question"]}')
print(f'A: {qa_pairs[0]["answer"]}')

Loaded 20 QA pairs

First QA pair:
Q: What is the standard architecture for building large language models?
A: The standard architecture for building large language models is the transformer.


## 2. Create Embeddings & FAISS Vector Store

In [4]:
def get_embedding(text):
    """Get embedding from OpenAI text-embedding-3-small"""
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

In [5]:
# Create embeddings for all chunks
print("Creating embeddings for all chunks...")
embeddings = []

for i, chunk in enumerate(chunks):
    emb = get_embedding(chunk['text'])
    embeddings.append(emb)
    
    # Progress update every 50 chunks
    if (i + 1) % 50 == 0 or i + 1 == len(chunks):
        print(f'Embedded chunk {i+1}/{len(chunks)}', end='\r')

print(f'\n✓ Created {len(embeddings)} embeddings')
print(f'Embedding dimension: {len(embeddings[0])}')

Creating embeddings for all chunks...
Embedded chunk 170/170
✓ Created 170 embeddings
Embedding dimension: 1536


In [6]:
# Create FAISS index
dimension = len(embeddings[0])
index = faiss.IndexFlatL2(dimension)

# Convert to numpy array and add to index
embeddings_array = np.array(embeddings).astype('float32')
index.add(embeddings_array)

print(f'✓ FAISS index created with {index.ntotal} vectors')

✓ FAISS index created with 170 vectors


In [7]:
# Save FAISS index
os.makedirs('vectorstore/naive_rag', exist_ok=True)
faiss.write_index(index, 'vectorstore/naive_rag/index.faiss')

print('✓ Saved FAISS index to vectorstore/naive_rag/index.faiss')

✓ Saved FAISS index to vectorstore/naive_rag/index.faiss


## 3. Retrieval Pipeline

In [8]:
def retrieve(query, k=3):
    """Retrieve top-k most relevant chunks"""
    # Get query embedding
    query_embedding = get_embedding(query)
    query_vector = np.array([query_embedding]).astype('float32')
    
    # Search in FAISS index
    distances, indices = index.search(query_vector, k)
    
    # Format results
    retrieved = []
    for idx, dist in zip(indices[0], distances[0]):
        retrieved.append({
            'chunk': chunks[idx]['text'],
            'page': chunks[idx]['page'],
            'chunk_id': chunks[idx]['chunk_id'],
            'distance': float(dist),
            'similarity': 1 / (1 + float(dist))  # Convert distance to similarity
        })
    
    return retrieved

In [9]:
# Test retrieval
test_query = "What is self-attention?"
retrieved_chunks = retrieve(test_query, k=3)

print(f"Query: {test_query}\n")
print("Retrieved chunks:")
for i, chunk in enumerate(retrieved_chunks):
    print(f"\n{i+1}. Page {chunk['page']} (Similarity: {chunk['similarity']:.3f})")
    print(f"   {chunk['chunk'][:200]}...")

Query: What is self-attention?

Retrieved chunks:

1. Page 1 (Similarity: 0.556)
   also called a self-attention layer. Attention can be thought of as a way to build
contextual representations of a token’s meaning by attending to and integrating
information from surrounding tokens, h...

2. Page 4 (Similarity: 0.495)
   Layer
attention
attention
attention
a1
a2
a3
a4
a5
x3
x4
x5
x1
x2
Figure 8.4
Information ﬂow in causal self-attention. When processing each input xi, the
model attends to all the inputs up to, and inc...

3. Page 4 (Similarity: 0.486)
   of tokens) but no tokens after i. (By contrast, in Chapter 9 we’ll generalize attention
so it can also look ahead to future words.)
Fig. 8.4 illustrates this ﬂow of information in an entire causal sel...


## 4. Generation Pipeline

In [10]:
def generate_answer(question, retrieved_chunks):
    """Generate answer using GPT-4o-mini"""
    # Format context from retrieved chunks
    context = '\n\n'.join([
        f"[Page {chunk['page']}]: {chunk['chunk']}"
        for chunk in retrieved_chunks
    ])
    
    # Create prompt
    prompt = f"""You are a helpful assistant answering questions about Chapter 8: Transformers from the Stanford NLP textbook.

Use only the following context to answer the question. If you cannot answer based on the context, say so.

Context:
{context}

Question: {question}

Answer:"""
    
    # Call OpenAI API
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=500
    )
    
    return response.choices[0].message.content

In [11]:
# Test generation
test_answer = generate_answer(test_query, retrieved_chunks)

print(f"Question: {test_query}")
print(f"\nAnswer: {test_answer}")

Question: What is self-attention?

Answer: Self-attention is a mechanism that allows a model to build contextual representations of a token's meaning by attending to and integrating information from surrounding tokens. It helps the model learn how tokens relate to each other over large spans. In a self-attention layer, when processing each input token, the model attends to all the inputs up to and including that token, creating a weighted sum of context vectors based on the relationships between tokens. This process occurs in parallel at each token position, mapping input sequences to output sequences of the same length.


## 5. Evaluate on All QA Pairs

In [12]:
# Run Naive RAG on all QA pairs
results = []

print("Running Naive RAG on all QA pairs...\n")

for i, qa in enumerate(qa_pairs):
    # Retrieve relevant chunks
    retrieved = retrieve(qa['question'], k=3)
    
    # Generate answer
    answer = generate_answer(qa['question'], retrieved)
    
    # Store result
    results.append({
        'question': qa['question'],
        'ground_truth': qa['answer'],
        'naive_rag_answer': answer,
        'retrieved_chunks': [{
            'text': r['chunk'],
            'page': r['page'],
            'similarity': r['similarity']
        } for r in retrieved]
    })
    
    print(f'✓ Processed {i+1}/{len(qa_pairs)}: {qa["question"][:60]}...', end='\r')

print(f'\n\n✓ Completed {len(results)} QA pairs')

Running Naive RAG on all QA pairs...

✓ Processed 20/20: What concept distinguishes transformers from feedforward lay...

✓ Completed 20 QA pairs


In [13]:
# Save results
os.makedirs('results', exist_ok=True)

with open('results/naive_rag_responses.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print('✓ Saved results to results/naive_rag_responses.json')

✓ Saved results to results/naive_rag_responses.json


## 6. Preview Results

In [14]:
# Display first 3 results
print("First 3 Q&A Results:\n")

for i, result in enumerate(results[:3]):
    print(f"\n{'='*80}")
    print(f"Question {i+1}: {result['question']}")
    print(f"\nGround Truth: {result['ground_truth']}")
    print(f"\nNaive RAG Answer: {result['naive_rag_answer']}")
    print(f"\nRetrieved from pages: {[c['page'] for c in result['retrieved_chunks']]}")

First 3 Q&A Results:


Question 1: What is the standard architecture for building large language models?

Ground Truth: The standard architecture for building large language models is the transformer.

Naive RAG Answer: The standard architecture for building large language models is the Transformer architecture. This architecture typically includes stacked transformer blocks, which consist of multiple layers, attention heads, and mechanisms for processing input tokens to generate word probabilities. Transformers are designed to handle large context windows, allowing them to utilize extensive amounts of context for predicting upcoming words.

Retrieved from pages: [20, 25, 20]

Question 2: What is the purpose of transformers in language modeling?

Ground Truth: Transformers are used in language modeling to predict output tokens one by one by conditioning on the prior context.

Naive RAG Answer: Transformers serve as the standard architecture for building large language models, which hav

---

## Summary

In this notebook, we:
1. ✅ Loaded chunks and QA pairs from Phase 1
2. ✅ Created embeddings using OpenAI text-embedding-3-small
3. ✅ Built FAISS vector store (IndexFlatL2)
4. ✅ Implemented retrieval pipeline (top-3 chunks)
5. ✅ Implemented generation pipeline (GPT-4o-mini)
6. ✅ Evaluated on all 20 QA pairs
7. ✅ Saved results to JSON

**Next Steps:**
- Proceed to `03-contextual-retrieval-implementation.ipynb` to implement Contextual Retrieval